In [1]:
from pathlib import Path

import numpy as np
import time
import sys
import json
from tqdm.notebook import tqdm
import itertools

from SolverV9 import (
    LiouvilleSpectroscopySolver,
    SpectroscopyPlotter,
    standard_nq_protocol,
)

from IPython.display import display, HTML
display(HTML("<style>div.output_area pre, div.jp-OutputArea-output pre {white-space: pre;}</style>"))

np.set_printoptions(threshold=sys.maxsize, linewidth=np.inf, precision=5, suppress=True)

### Coupled Model

### Solver Parameters

- `T`: System temperature. `T = 0` assumes that the system initially occupies its ground state.
- `Eta`: Homogeneous broadening used in the resolvents, in eV. It primarily controls the spectral linewidth.
- `backend`: Numerical implementation used by the solver. `"dense"` uses dense matrices and NumPy linear-algebra routines.

In [2]:
solver_params = {
    "T": 0.0,
    "Eta": 0.000001,
    "backend": "dense",
    "parallel_backend": "threading",
    "n_jobs": -1,
}

In [3]:
hbar = 0.658211951  # in eV fs

# parameters
en_h=1.5245 # heavy-hole exciton resonant energy in eV
en_l=1.5287 # light-hole exciton resonant energy in eV
detuning = -0.002 # detuning between heavy-hole exciton and cavity in eV
en_cav=en_h + detuning # cavity resonant energy in eV

rabi_h = 0.00004  # vaccum rabi splitting for H
g_h = rabi_h / 2  # coupling strength for light-matter interaction in eV for H
rabi_l = rabi_h / np.sqrt(3)  # vaccum rabi splitting for L
g_l = rabi_l / 2  # coupling strength for light-matter interaction in eV for L
J = 0.00002 # coupling strength between H and L
n_dens = 0. # density term for pauli blocking
n_th = 0. # average number of cavity photons

mu_qm1 = 1 # dipole strength for C
mu_qm2 = 1  # dipole strength for CC
mu_h1 = 1  # dipole strength for H
mu_h2 = 1  # dipole strength for 2Q H
mu_l1 = 0.#mu_h1 / np.sqrt(3)  # dipole strength for L
mu_l2 = 0.#mu_h2 / np.sqrt(3)  # dipole strength for 2Q L

kappa = 0.0011 # cavity decay strength
gamma_h_decay = 0.0001  # exciton decay strength
gamma_l_decay = 0.0001  # exciton decay strength
gamma_cav_phase = 0.00001 # cavity dephasing strength
gamma_h_phase = 0.00001 # exciton dephasing strength
gamma_l_phase = 0.00001 # exciton dephasing strength
# gamma_xc = 0.001 # x-c swapping strength
# gamma_cx = 0.001 # c-x swapping strength

hhb_bind=0.00045 # strength of HH_b biexciton binding
hlb_bind=0.000225 # strength of HL_b biexciton binding
llb_bind=0.00015 # strength of LL_b biexciton binding

grid = 100
t2 = 50

date = time.strftime('%Y-%m-%d')
hour = time.strftime("%Hh%Mm%S")

### 0-quantum + 1-quantum + 2-quantum
loop over i for any parameter

In [4]:
param = "test"
rang = ""
# rang = [0.04,0.3,5]
# kappa_list = [0.0011, 0.0015, 0.0019]
# for i in tqdm(np.linspace(*rang)):
# for i in tqdm(np.geomspace(*rang)):
# for i in tqdm(list(itertools.product(np.linspace(*rang), np.linspace(*rang)))):
# for i in tqdm(list(itertools.product(np.linspace(*rang), np.linspace(*rang), np.linspace(rang[0], rang[1] * 2, rang[2])))):
# for i in tqdm(list(itertools.product(np.linspace(*rang), np.linspace(*rang), np.linspace(*rang), np.linspace(*rang)))):
for i in [1]:
    # mu_qm1 = i[0]
    # mu_qm2 = i[1]
    # mu_x1 = i[2]
    # mu_x2 = i[3]
#     g = i
#     gamma_cav_phase = i[2]
#     en_cav=en_b + detuning
#     print(param_name)


    # setting up dm
    exc_basis = np.zeros((13, 13))
    exc_basis[0, 0] = 1 # G,H,L,C,HH_f,HH_b,HL_f,HL_b,LL_f,LL_b,HC,LC,CC
    rho = exc_basis @ exc_basis.T  # ground state of Hamiltonian

    wc = en_cav / hbar
    wh = en_h / hbar
    wl = en_l / hbar

    # each individual lowering operators
    h_low = np.zeros_like(exc_basis)
    h_low[0, 1] = 1

    l_low = np.zeros_like(exc_basis)
    l_low[0, 2] = 1

    c_low = np.zeros_like(exc_basis)
    c_low[0, 3] = 1

    hhf_low = np.zeros_like(exc_basis)
    hhf_low[1, 4] = 1

    hhb_low = np.zeros_like(exc_basis)
    hhb_low[1, 5] = 1

    hlf_low = np.zeros_like(exc_basis)
    hlf_low[1, 6] = 1 / np.sqrt(2)

    hlb_low = np.zeros_like(exc_basis)
    hlb_low[1, 7] = 1 / np.sqrt(2)

    lhf_low = np.zeros_like(exc_basis)
    lhf_low[2, 6] = 1 / np.sqrt(2)

    lhb_low = np.zeros_like(exc_basis)
    lhb_low[2, 7] = 1 / np.sqrt(2)

    llf_low = np.zeros_like(exc_basis)
    llf_low[2, 8] = 1

    llb_low = np.zeros_like(exc_basis)
    llb_low[2, 9] = 1

    hc_low = np.zeros_like(exc_basis)
    hc_low[1, 10] = 1 / np.sqrt(2)

    ch_low = np.zeros_like(exc_basis)
    ch_low[3, 10] = 1 / np.sqrt(2)

    lc_low = np.zeros_like(exc_basis)
    lc_low[2, 11] = 1 / np.sqrt(2)

    cl_low = np.zeros_like(exc_basis)
    cl_low[3, 11] = 1 / np.sqrt(2)

    cc_low = np.zeros_like(exc_basis)
    cc_low[3, 12] = 1

    # global lowering operators
    cav_low = c_low + hc_low + lc_low + cc_low
    exc_h_low = h_low + hhf_low + hhb_low + lhf_low + lhb_low + ch_low
    exc_l_low = l_low + llf_low + llb_low + hlf_low + hlb_low + cl_low

    H0_h = h_low.T @ h_low  # h-pop
    H0_l = l_low.T @ l_low  # xl-pop
    H0_c = c_low.T @ c_low  # c-pop
    H0_hhf = hhf_low.T @ hhf_low  # hhf-pop
    H0_hhb = hhb_low.T @ hhb_low  # hhf-pop
    H0_hlf = hlf_low.T @ hlf_low + lhf_low.T @ lhf_low  # hlf-pop
    H0_hlb = hlb_low.T @ hlb_low + lhb_low.T @ lhb_low  # hlb-pop
    H0_llf = llf_low.T @ llf_low  # hhf-pop
    H0_llb = llb_low.T @ llb_low  # hhb-pop
    H0_hc = hc_low.T @ hc_low + ch_low.T @ ch_low  # hc-pop
    H0_lc = lc_low.T @ lc_low + cl_low.T @ cl_low  # lc-pop
    H0_cc = cc_low.T @ cc_low  # cc-pop

    H0 = (en_h * H0_h + en_l * H0_l + en_cav * H0_c + 2 * en_h * H0_hhf + (2 * en_h - hhb_bind) * H0_hhb +
          (en_h + en_l) * H0_hlf + (en_h + en_l - hlb_bind) * H0_hlb + 2 * en_l * H0_llf +
          (2 * en_l - llb_bind) * H0_llb + (en_h + en_cav) * H0_hc + (en_l + en_cav) * H0_lc + 2 * en_cav * H0_cc)

    H_int_h = h_low.T @ c_low  # 1Q light-matter coupling
    H_int_h += 2 * ((1 - n_dens) * (hhf_low.T + hhb_low.T) @ hc_low + (lhf_low.T + lhb_low.T) @ lc_low + ch_low.T @ cc_low) # 2Q heavy light-matter coupling
    H_int_h += H_int_h.T

    H_int_l = l_low.T @ c_low  # 1Q light-matter coupling
    H_int_l += 2 * ((1 - n_dens) * (llf_low.T + llb_low.T) @ lc_low + (hlf_low.T + hlb_low.T) @ hc_low + cl_low.T @ cc_low) # 2Q light light-matter coupling
    H_int_l += H_int_l.T

    H_coupling = h_low.T @ l_low
    H_coupling += 2 * (1 - n_dens) * ((hhf_low.T + hhb_low.T) @ (hlf_low + hlb_low) + (lhf_low.T + lhb_low.T) @ (llf_low + llb_low)) + 2 * ch_low.T @ cl_low
    H_coupling += H_coupling.T

    H = H0 + g_h * H_int_h + g_l * H_int_l + J * H_coupling  # total hamiltonian
    print("H0", H0)
    print("H_int\n", H_int_h + H_int_l)
    print("H_coupling\n", H_coupling)
    print("H", H)


    ad = mu_qm2 * cav_low + (mu_qm1 - mu_qm2) * c_low + mu_h2 * exc_h_low + (mu_h1 - mu_h2) * h_low + mu_l2 * exc_l_low + (mu_l1 - mu_l2) * l_low # lowering operator
    # ad = mu_cav * cav_low + mu_exc * ((1 - n_dens) * (exc_h_low + exc_l_low) + n_dens * (h_low + ch_low + l_low + cl_low)) # lowering operator
    obs = ad + ad.T
    mu = obs

    if isinstance(i, list) and i == rang[0] or i == 1:
        print("mud\n", mu)

    # collapse operators
    c_c_rel = np.sqrt(kappa * (n_th + 1)) * c_low # cavity relaxation
    c_c_exc = np.sqrt(kappa * n_th) * c_low.T # cavity excitation
    c_h_rel = np.sqrt(gamma_h_decay * (n_th + 1)) * h_low # exciton relaxation
    c_h_exc = np.sqrt(gamma_h_decay * n_th) * h_low.T # exciton excitation
    c_l_rel = np.sqrt(gamma_l_decay * (n_th + 1)) * l_low # exciton relaxation
    c_l_exc = np.sqrt(gamma_l_decay * n_th) * l_low.T # exciton excitation
    c_hhf_rel = np.sqrt(gamma_h_decay * (n_th + 1)) * hhf_low # exciton relaxation
    c_hhf_exc = np.sqrt(gamma_h_decay * n_th) * hhf_low.T # exciton excitation
    c_hhb_rel = np.sqrt(gamma_h_decay * (n_th + 1)) * hhb_low # exciton relaxation
    c_hhb_exc = np.sqrt(gamma_h_decay * n_th) * hhb_low.T # exciton excitation
    c_hlf_rel = np.sqrt(gamma_l_decay * (n_th + 1)) * hlf_low # cavity relaxation
    c_hlf_exc = np.sqrt(gamma_l_decay * n_th) * hlf_low.T # cavity excitation
    c_lhf_rel = np.sqrt(gamma_h_decay * (n_th + 1)) * lhf_low # exciton relaxation
    c_lhf_exc = np.sqrt(gamma_h_decay * n_th) * lhf_low.T # exciton excitation
    c_hlb_rel = np.sqrt(gamma_l_decay * (n_th + 1)) * hlb_low # cavity relaxation
    c_hlb_exc = np.sqrt(gamma_l_decay * n_th) * hlb_low.T # cavity excitation
    c_lhb_rel = np.sqrt(gamma_h_decay * (n_th + 1)) * lhb_low # exciton relaxation
    c_lhb_exc = np.sqrt(gamma_h_decay * n_th) * lhb_low.T # exciton excitation
    c_llf_rel = np.sqrt(gamma_l_decay * (n_th + 1)) * llf_low # exciton relaxation
    c_llf_exc = np.sqrt(gamma_l_decay * n_th) * llf_low.T # exciton excitation
    c_llb_rel = np.sqrt(gamma_l_decay * (n_th + 1)) * llb_low # exciton relaxation
    c_llb_exc = np.sqrt(gamma_l_decay * n_th) * llb_low.T # exciton excitation
    c_hc_rel = np.sqrt(kappa * (n_th + 1)) * hc_low # cavity relaxation
    c_hc_exc = np.sqrt(kappa * n_th) * hc_low.T # cavity excitation
    c_ch_rel = np.sqrt(gamma_h_decay * (n_th + 1)) * ch_low # exciton relaxation
    c_ch_exc = np.sqrt(gamma_h_decay * n_th) * ch_low.T # exciton excitation
    c_lc_rel = np.sqrt(kappa * (n_th + 1)) * lc_low # cavity relaxation
    c_lc_exc = np.sqrt(kappa * n_th) * lc_low.T # cavity excitation
    c_cl_rel = np.sqrt(gamma_l_decay * (n_th + 1)) * cl_low # exciton relaxation
    c_cl_exc = np.sqrt(gamma_l_decay * n_th) * cl_low.T # exciton excitation
    c_cc_rel = np.sqrt(kappa * (n_th + 1)) * cc_low # cavity relaxation
    c_cc_exc = np.sqrt(kappa * n_th) * cc_low.T # cavity excitation

    c_h_dep = np.sqrt(gamma_h_phase) * en_h * H0_h # H pop
    c_l_dep = np.sqrt(gamma_l_phase) * en_l * H0_l # H pop
    c_c_dep = np.sqrt(gamma_cav_phase) * en_cav * H0_c # C pop
    c_hhf_dep = np.sqrt(gamma_h_phase) * 2 * en_h * H0_hhf # HHf pop
    c_hhb_dep = np.sqrt(gamma_h_phase) * (2 * en_h - hhb_bind) * H0_hhb # HHf pop
    c_hlf_dep = (np.sqrt(gamma_h_phase) + np.sqrt(gamma_l_phase)) * (en_h + en_l) / 2 * H0_hlf # HLf pop
    c_hlb_dep = (np.sqrt(gamma_h_phase) + np.sqrt(gamma_l_phase)) * (en_h + en_l - hlb_bind) / 2 * H0_hlb # HLf pop
    c_llf_dep = np.sqrt(gamma_l_phase) * 2 * en_l * H0_llf # LLf pop
    c_llb_dep = np.sqrt(gamma_l_phase) * (2 * en_l - llb_bind) * H0_llb # LLf pop
    c_hc_dep = (np.sqrt(gamma_cav_phase) + np.sqrt(gamma_h_phase)) * (en_h + en_cav) / 2 * H0_hc # HC pop
    c_lc_dep = (np.sqrt(gamma_cav_phase) + np.sqrt(gamma_l_phase)) * (en_l + en_cav) / 2 * H0_lc # HC pop
    c_cc_dep = np.sqrt(gamma_cav_phase) * 2 * en_cav * H0_cc # CC pop
    c_dep = c_h_dep + c_l_dep + c_c_dep + c_hhf_dep + c_hhb_dep + c_hlf_dep + c_hlb_dep + c_llf_dep + c_llb_dep + c_hc_dep + c_lc_dep + c_cc_dep

    # c_x_cav = np.sqrt(gamma_xc) * cav_low.T @ exc_low # bright exciton to cavity photon transition
    # c_cav_x = np.sqrt(gamma_xc) * exc_low.T @ cav_low # cavity photon to bright exciton transition


    # c_ops = [(c_c_rel,1), (c_h_rel,1), (c_l_rel,1), (c_hhf_rel,1), (c_hhb_rel,1), (c_hlf_rel,1), (c_hlb_rel,1), (c_lhf_rel,1), (c_lhb_rel,1), (c_llf_rel,1), (c_llb_rel,1), (c_hc_rel,1), (c_ch_rel,1), (c_lc_rel,1), (c_cl_rel,1), (c_cc_rel,1), (c_h_dep,1), (c_l_dep,1), (c_l_dep,1), (c_hhf_dep,1), (c_hhb_dep,1), (c_hlf_dep,1), (c_hlb_dep,1), (c_llf_dep,1), (c_llb_dep,1), (c_hc_dep,1), (c_lc_dep,1), (c_cc_dep,1)]
    c_ops = []#(c_c_rel,1), (c_h_rel,1), (c_l_rel,1), (c_hhf_rel,1), (c_hhb_rel,1), (c_hlf_rel,1), (c_hlb_rel,1), (c_lhf_rel,1), (c_lhb_rel,1), (c_llf_rel,1), (c_llb_rel,1), (c_hc_rel,1), (c_ch_rel,1), (c_lc_rel,1), (c_cl_rel,1), (c_cc_rel,1), (c_dep,1)]
    # for j in c_ops:
    #     print()
    #     print(j)

    # solver
    solver = LiouvilleSpectroscopySolver(solver_params)
    solver.feed_model(
        H,
        mu,
        c_ops_raw=c_ops,
        initial_density_matrix=rho,
        density_matrix_basis="site",
    )

    # pathways
    pathways = solver.generate_pathways_with_ufss(
        "-++",
        maximum_manifold=2,
        component="chi3_1q",
    )

    protocol = standard_nq_protocol(
        order=1,
        nq_interval=1,
        detection_interval=3,
        n_interactions=3,
        nq_axis="omega_1q",
        detection_axis="omega_emit",
    )

    [(pathway.name, pathway.interactions, pathway.coherence_orders) for pathway in pathways]

    # generate the spectrum
    omega_1q = np.linspace(-1.5375, -1.5175, grid)
    omega_emit = np.linspace(1.5175, 1.5375, grid)

    delays = {
        "t2": t2 / hbar
    }

    result = solver.generate_NQ_spectrum(
        1,
        protocol,
        axes={"omega_1q": omega_1q, "omega_emit": omega_emit},
        delays=delays,
        pathways=pathways
        )

    save_pdf = True
    output_directory = Path.cwd() / "Result_Test" / "Multiorder_pathway_plot" / date / f"{hour}_{param}{rang}"
    output_directory.mkdir(parents=True, exist_ok=True)

    if isinstance(i, (int, float)):
        spec_real = f"chi3_Q1_real_{param}{i:.6f}.png"
        spec_imag = f"chi3_Q1_imag_{param}{i:.6f}.png"
        spec_abs = f"chi3_Q1_abs_{param}{i:.6f}.png"
    else:
        spec_real = f"chi3_Q1_real_{param}" + "_".join(f"{x:.6f}" for x in i) + ".png"
        spec_imag = f"chi3_Q1_imag_{param}" + "_".join(f"{x:.6f}" for x in i) + ".png"
        spec_abs = f"chi3_Q1_abs_{param}" + "_".join(f"{x:.6f}" for x in i) + ".png"

    plotter = SpectroscopyPlotter(detection_phase=np.pi/2)
    plot_result = plotter.plot_pathways_multiorder(
        result,
        pathways="all",
        totals=["1Q"],
        view="real",
        normalization="none",
        axis_labels={
            "omega_1q": "1Q energy (eV)",
            "omega_emit": "Emission energy (eV)",
        },
        include_diagrams=False,
        display_diagrams=False,
        save_pdf=save_pdf,
        output_directory=output_directory if save_pdf else None,
        spectrum_pdf_name = spec_real,
        show=False,
    )

    plot_result = plotter.plot_pathways_multiorder(
        result,
        pathways="all",
        totals=["1Q"],
        view="imag",
        normalization="none",
        axis_labels={
            "omega_1q": "1Q energy (eV)",
            "omega_emit": "Emission energy (eV)",
        },
        include_diagrams=False,
        display_diagrams=False,
        save_pdf=save_pdf,
        output_directory=output_directory if save_pdf else None,
spectrum_pdf_name = spec_imag,
        show=False,
    )

    plot_result = plotter.plot_pathways_multiorder(
        result,
        pathways="all",
        totals=["1Q"],
        view="abs",
        normalization="none",
        axis_labels={
            "omega_1q": "1Q energy (eV)",
            "omega_emit": "Emission energy (eV)",
        },
        include_diagrams=False,
        display_diagrams=False,
        save_pdf=save_pdf,
        output_directory=output_directory if save_pdf else None,
        spectrum_pdf_name = spec_abs,
        show=False,
    )

    # plotter.plot_spectrum_result_contours(
    #     result,
    #     spectra="pathways",
    #     names=["R1", "R2", "R3"],
    #     totals="selected",
    #     normalization="panel",
    #     color_map="PuOr",
    #     levels=25,
    #     show=False,
    # )
    print(f"\n{i} diagram done\n")

H0 [[0.      0.      0.      0.      0.      0.      0.      0.      0.      0.      0.      0.      0.     ]
 [0.      1.5245  0.      0.      0.      0.      0.      0.      0.      0.      0.      0.      0.     ]
 [0.      0.      1.5287  0.      0.      0.      0.      0.      0.      0.      0.      0.      0.     ]
 [0.      0.      0.      1.5225  0.      0.      0.      0.      0.      0.      0.      0.      0.     ]
 [0.      0.      0.      0.      3.049   0.      0.      0.      0.      0.      0.      0.      0.     ]
 [0.      0.      0.      0.      0.      3.04855 0.      0.      0.      0.      0.      0.      0.     ]
 [0.      0.      0.      0.      0.      0.      3.0532  0.      0.      0.      0.      0.      0.     ]
 [0.      0.      0.      0.      0.      0.      0.      3.05297 0.      0.      0.      0.      0.     ]
 [0.      0.      0.      0.      0.      0.      0.      0.      3.0574  0.      0.      0.      0.     ]
 [0.      0.      0.      0.      

### saving parameters and matrices

In [5]:
with open(f"{output_directory}/parameters.txt", "w", encoding="utf-8") as file:
    file.write(hour+"\n")
    file.write(f"Energies  heavy-hole={en_h:.7f}, light-hole={en_l:.7f}, cavity={en_cav:.7f}\n")
    file.write(f"detuning delta={detuning:.4}\n")
    file.write(f"light-matter coupling g_h={g_h:.7f}, g_l={g_l:.7f}\n")
    file.write(f"dephasing  heavy-hole={gamma_h_phase:.7f}, light-hole={gamma_l_phase:.7f}, cavity={gamma_cav_phase:.7f}\n")
    file.write(f"decay  heavy-hole={gamma_h_decay:.7f}, light-hole={gamma_l_decay:.7f}, cavity={kappa:.7f}\n")
    file.write(f"binding energies hhb={hhb_bind:.7f}, hlb={hlb_bind:.7f}, lb={llb_bind:.7f}\n")
    # file.write(f"swapping strength  exciton-cavity={gamma_xc:.4f}, cavity-exciton={gamma_cx:.4f}\n")
    file.write(f"broadening eta={solver_params["Eta"]}\n")
    file.write(f"grid size(f1,f3)={grid}\n")
    file.write(f"population time={t2}\n")
    file.write(f"average photons={n_th}\n")
    for ii in [H, H_int_h, H_int_l, H_coupling, mu]+[cc[0] for cc in c_ops]:
        for name, val in list(locals().items()):
            if val is ii:
                file.write(f"\n\n{name}\n")
                break
        for i, row in enumerate(ii.tolist()):
            formatted = [f"{v.real:.5f}" if v.real >= 0.000001 else "0" for v in row]
            file.write("[" + ", ".join(f"{s:>{8}}" for s in formatted) + "]" + "\n")
    # file.write("\n\nH_int\n")
    # for i, row in enumerate(H_int.tolist()):
    #     formatted = [f"{v:.5f}" if v != 0 else "0" for v in row]
    #     file.write("[" + ", ".join(f"{s:>{8}}" for s in formatted) + "]" + "\n")
    # file.write("\n\nH_coupling\n")
    # for i, row in enumerate(H_coupling.tolist()):
    #     formatted = [f"{v:.5f}" if v != 0 else "0" for v in row]
    #     file.write("[" + ", ".join(f"{s:>{8}}" for s in formatted) + "]" + "\n")
    # file.write("\n\nmu/obs\n")
    # for i, row in enumerate(mu.tolist()):
    #     formatted = [f"{v:.5f}" if v != 0 else "0" for v in row]
    #     file.write("[" + ", ".join(f"{s:>{8}}" for s in formatted) + "]" + "\n")
    # for j in c_ops:
    #     file.write("\n\n"+f"{j[0]=}".split("=")[0]+"\n")
    #     for i, row in enumerate(j[0].tolist()):
    #         formatted = [f"{v:.5f}" if v != 0 else "0" for v in row]
    #         file.write("[" + ", ".join(f"{s:>{8}}" for s in formatted) + "]" + "\n")

In [6]:
# print("Plotted panels:", plot_result.panel_names)
# print("Matching UFSS diagrams:", tuple(plot_result.diagrams))
# if save_pdf:
#     print("Spectrum PDF:", plot_result.spectrum_pdf)
#     print("Diagram PDFs:", plot_result.diagram_paths)
